In [1]:
import polars as pl
import re
import pathlib
import requests
import typing, gzip, os
from requests.adapters import HTTPAdapter, Retry
retries = Retry(total=5, backoff_factor=0.25, status_forcelist=[500, 502, 503, 504])
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retries))



In [2]:
CODON_MAP = {
    'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L', 
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S', 
    'TAT': 'Y', 'TAC': 'Y', 'TGT': 'C', 'TGC': 'C', 
    'TGG': 'W', 'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 
    'CTG': 'L', 'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 
    'CCG': 'P', 'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 
    'CAG': 'Q', 'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 
    'CGG': 'R', 'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 
    'ATG': 'M', 'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 
    'ACG': 'T', 'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 
    'AAG': 'K', 'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 
    'AGG': 'R', 'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 
    'GTG': 'V', 'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 
    'GCG': 'A', 'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 
    'GAG': 'E', 'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 
    'GGG': 'G',
    'TAA': '*', 'TAG': '*', 'TGA': '*'
}

SGC_STOCKHOLM_SRC_COLUMNS = {
    'Construct ID': 'construct_id',
    'DNA Sequence': 'dna_seq',
    'Expression ID': 'expression_id',
    'Insoluble': 'insoluble',
    'Soluble': 'soluble',
    'Yield': 'yield_cat',
    'Comments': 'comments',
    'Record Creator': 'record_creator'
}

def pl_expr_seq_translate(col_expr:pl.Expr)->pl.Expr:
    return (
        col_expr.str.to_uppercase()
        .str.extract_all(r'(.{3})')
        .list.eval(pl.element().replace_strict(CODON_MAP, default='X'))
        .list.join('')
    )


In [3]:
df = pl.read_parquet("~/data/pp/ai/datasets/17_01/full.pqt").filter(source='SGC_Stockholm', unique_target_count=1)
alignments_df = pl.read_csv('~/data/pp/ai/datasets/17_01/full_alignments.csv.gz')
sgc_stockholm_df = pl.read_parquet('~/data/pp/ai/datasets/17_01/sgc-stockholm/sgc_stockholm.pqt')
sgc_stockholm_src_df = pl.read_csv('~/data/pp/ai/datasets/17_01/sgc-stockholm/sgc_stockholm_source.csv.gz')
uniprot_localisation_df_file = pathlib.Path('~/Documents/nicola_paper_2026/sgc_stockholm_expert_uniprot.parquet').expanduser()
# df = df.join(sgc_stockholm_df.select('id', 'yield_cat', 'insoluble', 'soluble'), on='id', how='inner')


In [4]:
# https://www.ncbi.nlm.nih.gov/nuccore/EF198106.1
pNic28_BSA4_seq="""
TAATACGACTCACTATAGGGGAATTGTGAGCGGATAACAATTCCCCTCTAGAAATAATTTTGTTTAACTT
TAAGAAGGAGATATACATATGCACCATCATCATCATCATTCTTCTGGTGTAGATCTGGGTACCGAGAACC
TGTACTTCCAATCCATGGAGACCGACGTCCACATATACCTGCCGTTCACTATTATTTAGTGAAATGAGAT
ATTATGATATTTTCTGAATTGTGATTAAAAAGGCAACTTTATGCCCATGCAACAGAAACTATAAAAAATA
CAGAGAATGAAAAGAAACAGATAGATTTTTTAGTTCTTTAGGCCCGTAGTCTGCAAATCCTTTTATGATT
TTCTATCAAACAAAAGAGGAAAATAGACCAGTTGCAATCCAAACGAGAGTCTAATAGAATGAGGTCGAAA
AGTAAATCGCGCGGGTTTGTTACTGATAAAGCAGGCAAGACCTAAAATGTGTAAAGGGCAAAGTGTATAC
TTTGGCGTCACCCCTTACATATTTTAGGTCTTTTTTTATTGTGCGTAACTAACTTGCCATCTTCAAACAG
GAGGGCTGGAAGAAGCAGACCGCTAACACAGTACATAAAAAAGGAGACATGAACGATGAACATCAAAAAG
TTTGCAAAACAAGCAACAGTATTAACCTTTACTACCGCACTGCTGGCAGGAGGCGCAACTCAAGCGTTTG
CGAAAGAAACGAACCAAAAGCCATATAAGGAAACATACGGCATTTCCCATATTACACGCCATGATATGCT
GCAAATCCCTGAACAGCAAAAAAATGAAAAATATAAAGTTCCTGAGTTCGATTCGTCCACAATTAAAAAT
ATCTCTTCTGCAAAAGGCCTGGACGTTTGGGACAGCTGGCCATTACAAAACACTGACGGCACTGTCGCAA
ACTATCACGGCTACCACATCGTCTTTGCATTAGCCGGAGATCCTAAAAATGCGGATGACACATCGATTTA
CATGTTCTATCAAAAAGTCGGCGAAACTTCTATTGACAGCTGGAAAAACGCTGGCCGCGTCTTTAAAGAC
AGCGACAAATTCGATGCAAATGATTCTATCCTAAAAGACCAAACACAAGAATGGTCAGGTTCAGCCACAT
TTACATCTGACGGAAAAATCCGTTTATTCTACACTGATTTCTCCGGTAAACATTACGGCAAACAAACACT
GACAACTGCACAAGTTAACGTATCAGCATCAGACAGCTCTTTGAACATCAACGGTGTAGAGGATTATAAA
TCAATCTTTGACGGTGACGGAAAAACGTATCAAAATGTACAGCAGTTCATCGATGAAGGCAACTACAGCT
CAGGCGACAACCATACGCTGAGAGATCCTCACTACGTAGAAGATAAAGGCCACAAATACTTAGTATTTGA
AGCAAACACTGGAACTGAAGATGGCTACCAAGGCGAAGAATCTTTATTTAACAAAGCATACTATGGCAAA
AGCACATCATTCTTCCGTCAAGAAAGTCAAAAACTTCTGCAAAGCGATAAAAAACGCACGGCTGAGTTAG
CAAACGGCGCTCTCGGTATGATTGAGCTAAACGATGATTACACACTGAAAAAAGTGATGAAACCGCTGAT
TGCATCTAACACAGTAACAGATGAAATTGAACGCGCGAACGTCTTTAAAATGAACGGCAAATGGTACCTG
TTCACTGACTCCCGCGGATCAAAAATGACGATTGACGGCATTACGTCTAACGATATTTACATGCTTGGTT
ATGTTTCTAATTCTTTAACTGGCCCATACAAGCCGCTGAACAAAACTGGCCTTGTGTTAAAAATGGATCT
TGATCCTAACGATGTAACCTTTACTTACTCACACTTCGCTGTACCTCAAGCGAAAGGAAACAATGTCGTG
ATTACAAGCTATATGACAAACAGAGGATTCTACGCAGACAAACAATCAACGTTTGCGCCTAGCTTCCTGC
TGAACATCAAAGGCAAGAAAACATCTGTTGTCAAAGACAGCATCCTTGAACAAGGACAATTAACAGTTAA
CAAATAAAAACGCAAAAGAAAATGCCGATATCCTATTGGCATTGACGGTCTCCAGTAAAGGTGGATACGG
ATCCGAATTCGAGCTCCGTCGACAAGCTTGCGGCCGCACTCGAGCACCACCACCACCACCACTGAGATCC
GGCTGCTAACAAAGCCCGAAAGGAAGCTGAGTTGGCTGCTGCCACCGCTGAGCAATAACTAGCATAACCC
CTTGGGGCCTCTAAACGGGTCTTGAGGGGTTTTTTGCTGAAAGGAGGAACTATATCCGGATTGGCGAATG
GGACGCGCCCTGTAGCGGCGCATTAAGCGCGGCGGGTGTGGTGGTTACGCGCAGCGTGACCGCTACACTT
GCCAGCGCCCTAGCGCCCGCTCCTTTCGCTTTCTTCCCTTCCTTTCTCGCCACGTTCGCCGGCTTTCCCC
GTCAAGCTCTAAATCGGGGGCTCCCTTTAGGGTTCCGATTTAGTGCTTTACGGCACCTCGACCCCAAAAA
ACTTGATTAGGGTGATGGTTCACGTAGTGGGCCATCGCCCTGATAGACGGTTTTTCGCCCTTTGACGTTG
GAGTCCACGTTCTTTAATAGTGGACTCTTGTTCCAAACTGGAACAACACTCAACCCTATCTCGGTCTATT
CTTTTGATTTATAAGGGATTTTGCCGATTTCGGCCTATTGGTTAAAAAATGAGCTGATTTAACAAAAATT
TAACGCGAATTTTAACAAAATATTAACGTTTACAATTTCAGGTGGCACTTTTCGGGGAAATGTGCGCGGA
ACCCCTATTTGTTTATTTTTCTAAATACATTCAAATATGTATCCGCTCATGAATTAATTCTTAGAAAAAC
TCATCGAGCATCAAATGAAACTGCAATTTATTCATATCAGGATTATCAATACCATATTTTTGAAAAAGCC
GTTTCTGTAATGAAGGAGAAAACTCACCGAGGCAGTTCCATAGGATGGCAAGATCCTGGTATCGGTCTGC
GATTCCGACTCGTCCAACATCAATACAACCTATTAATTTCCCCTCGTCAAAAATAAGGTTATCAAGTGAG
AAATCACCATGAGTGACGACTGAATCCGGTGAGAATGGCAAAAGTTTATGCATTTCTTTCCAGACTTGTT
CAACAGGCCAGCCATTACGCTCGTCATCAAAATCACTCGCATCAACCAAACCGTTATTCATTCGTGATTG
CGCCTGAGCGAGACGAAATACGCGATCGCTGTTAAAAGGACAATTACAAACAGGAATCGAATGCAACCGG
CGCAGGAACACTGCCAGCGCATCAACAATATTTTCACCTGAATCAGGATATTCTTCTAATACCTGGAATG
CTGTTTTCCCGGGGATCGCAGTGGTGAGTAACCATGCATCATCAGGAGTACGGATAAAATGCTTGATGGT
CGGAAGAGGCATAAATTCCGTCAGCCAGTTTAGTCTGACCATCTCATCTGTAACATCATTGGCAACGCTA
CCTTTGCCATGTTTCAGAAACAACTCTGGCGCATCGGGCTTCCCATACAATCGATAGATTGTCGCACCTG
ATTGCCCGACATTATCGCGAGCCCATTTATACCCATATAAATCAGCATCCATGTTGGAATTTAATCGCGG
CCTAGAGCAAGACGTTTCCCGTTGAATATGGCTCATAACACCCCTTGTATTACTGTTTATGTAAGCAGAC
AGTTTTATTGTTCATGACCAAAATCCCTTAACGTGAGTTTTCGTTCCACTGAGCGTCAGACCCCGTAGAA
AAGATCAAAGGATCTTCTTGAGATCCTTTTTTTCTGCGCGTAATCTGCTGCTTGCAAACAAAAAAACCAC
CGCTACCAGCGGTGGTTTGTTTGCCGGATCAAGAGCTACCAACTCTTTTTCCGAAGGTAACTGGCTTCAG
CAGAGCGCAGATACCAAATACTGTCCTTCTAGTGTAGCCGTAGTTAGGCCACCACTTCAAGAACTCTGTA
GCACCGCCTACATACCTCGCTCTGCTAATCCTGTTACCAGTGGCTGCTGCCAGTGGCGATAAGTCGTGTC
TTACCGGGTTGGACTCAAGACGATAGTTACCGGATAAGGCGCAGCGGTCGGGCTGAACGGGGGGTTCGTG
CACACAGCCCAGCTTGGAGCGAACGACCTACACCGAACTGAGATACCTACAGCGTGAGCTATGAGAAAGC
GCCACGCTTCCCGAAGGGAGAAAGGCGGACAGGTATCCGGTAAGCGGCAGGGTCGGAACAGGAGAGCGCA
CGAGGGAGCTTCCAGGGGGAAACGCCTGGTATCTTTATAGTCCTGTCGGGTTTCGCCACCTCTGACTTGA
GCGTCGATTTTTGTGATGCTCGTCAGGGGGGCGGAGCCTATGGAAAAACGCCAGCAACGCGGCCTTTTTA
CGGTTCCTGGCCTTTTGCTGGCCTTTTGCTCACATGTTCTTTCCTGCGTTATCCCCTGATTCTGTGGATA
ACCGTATTACCGCCTTTGAGTGAGCTGATACCGCTCGCCGCAGCCGAACGACCGAGCGCAGCGAGTCAGT
GAGCGAGGAAGCGGAAGAGCGCCTGATGCGGTATTTTCTCCTTACGCATCTGTGCGGTATTTCACACCGC
ATATATGGTGCACTCTCAGTACAATCTGCTCTGATGCCGCATAGTTAAGCCAGTATACACTCCGCTATCG
CTACGTGACTGGGTCATGGCTGCGCCCCGACACCCGCCAACACCCGCTGACGCGCCCTGACGGGCTTGTC
TGCTCCCGGCATCCGCTTACAGACAAGCTGTGACCGTCTCCGGGAGCTGCATGTGTCAGAGGTTTTCACC
GTCATCACCGAAACGCGCGAGGCAGCTGCGGTAAAGCTCATCAGCGTGGTCGTGAAGCGATTCACAGATG
TCTGCCTGTTCATCCGCGTCCAGCTCGTTGAGTTTCTCCAGAAGCGTTAATGTCTGGCTTCTGATAAAGC
GGGCCATGTTAAGGGCGGTTTTTTCCTGTTTGGTCACTGATGCCTCCGTGTAAGGGGGATTTCTGTTCAT
GGGGGTAATGATACCGATGAAACGAGAGAGGATGCTCACGATACGGGTTACTGATGATGAACATGCCCGG
TTACTGGAACGTTGTGAGGGTAAACAACTGGCGGTATGGATGCGGCGGGACCAGAGAAAAATCACTCAGG
GTCAATGCCAGCGCTTCGTTAATACAGATGTAGGTGTTCCACAGGGTAGCCAGCAGCATCCTGCGATGCA
GATCCGGAACATAATGGTGCAGGGCGCTGACTTCCGCGTTTCCAGACTTTACGAAACACGGAAACCGAAG
ACCATTCATGTTGTTGCTCAGGTCGCAGACGTTTTGCAGCAGCAGTCGCTTCACGTTCGCTCGCGTATCG
GTGATTCATTCTGCTAACCAGTAAGGCAACCCCGCCAGCCTAGCCGGGTCCTCAACGACAGGAGCACGAT
CATGCGCACCCGTGGGGCCGCCATGCCGGCGATAATGGCCTGCTTCTCGCCGAAACGTTTGGTGGCGGGA
CCAGTGACGAAGGCTTGAGCGAGGGCGTGCAAGATTCCGAATACCGCAAGCGACAGGCCGATCATCGTCG
CGCTCCAGCGAAAGCGGTCCTCGCCGAAAATGACCCAGAGCGCTGCCGGCACCTGTCCTACGAGTTGCAT
GATAAAGAAGACAGTCATAAGTGCGGCGACGATAGTCATGCCCCGCGCCCACCGGAAGGAGCTGACTGGG
TTGAAGGCTCTCAAGGGCATCGGTCGAGATCCCGGTGCCTAATGAGTGAGCTAACTTACATTAATTGCGT
TGCGCTCACTGCCCGCTTTCCAGTCGGGAAACCTGTCGTGCCAGCTGCATTAATGAATCGGCCAACGCGC
GGGGAGAGGCGGTTTGCGTATTGGGCGCCAGGGTGGTTTTTCTTTTCACCAGTGAGACGGGCAACAGCTG
ATTGCCCTTCACCGCCTGGCCCTGAGAGAGTTGCAGCAAGCGGTCCACGCTGGTTTGCCCCAGCAGGCGA
AAATCCTGTTTGATGGTGGTTAACGGCGGGATATAACATGAGCTGTCTTCGGTATCGTCGTATCCCACTA
CCGAGATATCCGCACCAACGCGCAGCCCGGACTCGGTAATGGCGCGCATTGCGCCCAGCGCCATCTGATC
GTTGGCAACCAGCATCGCAGTGGGAACGATGCCCTCATTCAGCATTTGCATGGTTTGTTGAAAACCGGAC
ATGGCACTCCAGTCGCCTTCCCGTTCCGCTATCGGCTGAATTTGATTGCGAGTGAGATATTTATGCCAGC
CAGCCAGACGCAGACGCGCCGAGACAGAACTTAATGGGCCCGCTAACAGCGCGATTTGCTGGTGACCCAA
TGCGACCAGATGCTCCACGCCCAGTCGCGTACCGTCTTCATGGGAGAAAATAATACTGTTGATGGGTGTC
TGGTCAGAGACATCAAGAAATAACGCCGGAACATTAGTGCAGGCAGCTTCCACAGCAATGGCATCCTGGT
CATCCAGCGGATAGTTAATGATCAGCCCACTGACGCGTTGCGCGAGAAGATTGTGCACCGCCGCTTTACA
GGCTTCGACGCCGCTTCGTTCTACCATCGACACCACCACGCTGGCACCCAGTTGATCGGCGCGAGATTTA
ATCGCCGCGACAATTTGCGACGGCGCGTGCAGGGCCAGACTGGAGGTGGCAACGCCAATCAGCAACGACT
GTTTGCCCGCCAGTTGTTGTGCCACGCGGTTGGGAATGTAATTCAGCTCCGCCATCGCCGCTTCCACTTT
TTCCCGCGTTTTCGCAGAAACGTGGCTGGCCTGGTTCACCACGCGGGAAACGGTCTGATAAGAGACACCG
GCATACTCTGCGACATCGTATAACGTTACTGGTTTCACATTCACCACCCTGAATTGACTCTCTTCCGGGC
GCTATCATGCCATACCGCGAAAGGTTTTGCGCCATTCGATGGTGTCCGGGATCTCGACGCTCTCCCTTAT
GCGACTCCTGCATTAGGAAGCAGCCCAGTAGTAGGTTGAGGCCGTTGAGCACCGCCGCCGCAAGGAATGG
TGCATGCAAGGAGATGGCGCCCAACAGTCCCCCGGCCACGGGGCCTGCCACCATACCCACGCCGAAACAA
GCGCTCATGAGCCCGAAGTGGCGAGCCCGATCTTCCCCATCGGTGATGTCGGCGATATAGGCGCCAGCAA
CCGCACCTGTGGCGCCGGTGATGCCGGCCACGATGCGTCCGGCGTAGAGGATCGAGATCTCGATCCCGCG
AAAT"""
pNic28_BSA4_seq = re.compile(r'\s+').sub('', pNic28_BSA4_seq)
pNic28_BSA4_seq
# Email from Opher Gileadi
replace_BsaI='ATGCACCATCATCATCATCATTCTTCTGGTGTAGATCTGGGTACCGAGAACCTGTACTTCCAATCCATGGAGACCGACGTCCACATATACCTGCCGTTCACTATTATTTAGTGAAATGAGATATTATGATATTTTCTGAATTGTGATTAAAAAGGCAACTTTATGCCCATGCAACAGAAACTATAAAAAATACAGAGAATGAAAAGAAACAGATAGATTTTTTAGTTCTTTAGGCCCGTAGTCTGCAAATCCTTTTATGATTTTCTATCAAACAAAAGAGGAAAATAGACCAGTTGCAATCCAAACGAGAGTCTAATAGAATGAGGTCGAAAAGTAAATCGCGCGGGTTTGTTACTGATAAAGCAGGCAAGACCTAAAATGTGTAAAGGGCAAAGTGTATACTTTGGCGTCACCCCTTACATATTTTAGGTCTTTTTTTATTGTGCGTAACTAACTTGCCATCTTCAAACAGGAGGGCTGGAAGAAGCAGACCGCTAACACAGTACATAAAAAAGGAGACATGAACGATGAACATCAAAAAGTTTGCAAAACAAGCAACAGTATTAACCTTTACTACCGCACTGCTGGCAGGAGGCGCAACTCAAGCGTTTGCGAAAGAAACGAACCAAAAGCCATATAAGGAAACATACGGCATTTCCCATATTACACGCCATGATATGCTGCAAATCCCTGAACAGCAAAAAAATGAAAAATATAAAGTTCCTGAGTTCGATTCGTCCACAATTAAAAATATCTCTTCTGCAAAAGGCCTGGACGTTTGGGACAGCTGGCCATTACAAAACACTGACGGCACTGTCGCAAACTATCACGGCTACCACATCGTCTTTGCATTAGCCGGAGATCCTAAAAATGCGGATGACACATCGATTTACATGTTCTATCAAAAAGTCGGCGAAACTTCTATTGACAGCTGGAAAAACGCTGGCCGCGTCTTTAAAGACAGCGACAAATTCGATGCAAATGATTCTATCCTAAAAGACCAAACACAAGAATGGTCAGGTTCAGCCACATTTACATCTGACGGAAAAATCCGTTTATTCTACACTGATTTCTCCGGTAAACATTACGGCAAACAAACACTGACAACTGCACAAGTTAACGTATCAGCATCAGACAGCTCTTTGAACATCAACGGTGTAGAGGATTATAAATCAATCTTTGACGGTGACGGAAAAACGTATCAAAATGTACAGCAGTTCATCGATGAAGGCAACTACAGCTCAGGCGACAACCATACGCTGAGAGATCCTCACTACGTAGAAGATAAAGGCCACAAATACTTAGTATTTGAAGCAAACACTGGAACTGAAGATGGCTACCAAGGCGAAGAATCTTTATTTAACAAAGCATACTATGGCAAAAGCACATCATTCTTCCGTCAAGAAAGTCAAAAACTTCTGCAAAGCGATAAAAAACGCACGGCTGAGTTAGCAAACGGCGCTCTCGGTATGATTGAGCTAAACGATGATTACACACTGAAAAAAGTGATGAAACCGCTGATTGCATCTAACACAGTAACAGATGAAATTGAACGCGCGAACGTCTTTAAAATGAACGGCAAATGGTACCTGTTCACTGACTCCCGCGGATCAAAAATGACGATTGACGGCATTACGTCTAACGATATTTACATGCTTGGTTATGTTTCTAATTCTTTAACTGGCCCATACAAGCCGCTGAACAAAACTGGCCTTGTGTTAAAAATGGATCTTGATCCTAACGATGTAACCTTTACTTACTCACACTTCGCTGTACCTCAAGCGAAAGGAAACAATGTCGTGATTACAAGCTATATGACAAACAGAGGATTCTACGCAGACAAACAATCAACGTTTGCGCCTAGCTTCCTGCTGAACATCAAAGGCAAGAAAACATCTGTTGTCAAAGACAGCATCCTTGAACAAGGACAATTAACAGTTAACAAATAAAAACGCAAAAGAAAATGCCGATATCCTATTGGCATTGACGGTCTCC'

In [5]:
(pNic28_BSA4_seq.index(replace_BsaI), pNic28_BSA4_seq.index(replace_BsaI) + len(replace_BsaI))

(88, 2083)

In [6]:
sgc_stockholm_fixed_df = (
    sgc_stockholm_src_df
    .rename(SGC_STOCKHOLM_SRC_COLUMNS)
    .filter(pl.col.dna_seq.is_not_null(), pl.col.yield_cat.is_not_null())
    .with_columns(pl.col.dna_seq.str.slice(0, pl.col.dna_seq.str.len_bytes() // 3 * 3))
    .with_columns(protein_seq=pl_expr_seq_translate(pl.col.dna_seq))
    .with_columns(
        protein_seq_start=pl.when(
            pl.col.protein_seq.str.contains('^H+M') | 
            pl.col.protein_seq.str.starts_with('KRR*LM') | 
            pl.col.protein_seq.str.starts_with('LRRRYTM')
        ).then(pl.col.protein_seq.str.find('M')).otherwise(0),
    )
    .with_columns(
        dna_seq_start=pl.col.protein_seq_start * 3,
        protein_stop_codon_pos=pl.col.protein_seq_start + 
            pl.col.protein_seq.str.slice(pl.col.protein_seq_start).str.find('*', literal=True)
    )
    .with_columns(
        dna_seq_end=3 * (pl.col.protein_seq_start + pl.col.protein_stop_codon_pos + 
                    pl.col.protein_seq.str.slice(pl.col.protein_seq_start + pl.col.protein_stop_codon_pos).str.find('[^*]'))
    )
    .with_columns(dna_seq_fixed=pl.col.dna_seq.str.slice(pl.col.dna_seq_start, pl.col.dna_seq_end - pl.col.dna_seq_start))
    .filter(pl.col.dna_seq_fixed.str.starts_with('ATG'))
    .with_columns(protein_seq_translated=pl_expr_seq_translate(pl.col.dna_seq_fixed))
    .filter(pl.col.protein_seq_translated.str.slice(-1) == '*')
)
# check that dna sequence is unique per construct id
assert sgc_stockholm_fixed_df.group_by('construct_id').agg(pl.col.dna_seq_fixed.n_unique()).select((pl.col.dna_seq_fixed > 1).sum()).item() == 0
# check that ('construct_id', 'expression_id') id unique
assert sgc_stockholm_fixed_df.group_by('construct_id', 'expression_id').len().select((pl.col.len == 1).all()).item()
# check that protein sequences match
assert (df.select('id', 'protein_seq')
    .join(sgc_stockholm_df.select('id', 'construct_id', 'expression_id'), on='id', how='inner')
    .join(sgc_stockholm_fixed_df.select('construct_id',  'expression_id', 'protein_seq_translated'), on=('construct_id', 'expression_id'), how='inner')
    .select((pl.col.protein_seq.str.replace(r'\*+$', '') == pl.col.protein_seq).all())
    .item()
)
df = (
    df.select('id', 'fasta_id', 'dna_fasta_id', 'yield_binary', 'uniprot_id', 'taxon_id', 'gene_id')
    .with_columns(pl.col.gene_id.replace({'nan': None}), vector_seq=pl.lit(pNic28_BSA4_seq))
    .join(sgc_stockholm_df.select('id', 'construct_id', 'expression_id', 'yield_cat'), on='id', how='inner')
    .join(sgc_stockholm_fixed_df.select('construct_id',  'expression_id', 'protein_seq_translated', 'dna_seq_fixed'), on=('construct_id', 'expression_id'), how='inner')
    .rename({'protein_seq_translated': 'protein_seq', 'dna_seq_fixed': 'dna_seq'})
    .with_columns(
        pl.col.protein_seq.str.replace(r'\*+$', ''),
        pl.col.vector_seq.str.replace_all(pl.lit(replace_BsaI), pl.col.dna_seq, literal=True)
    )
    .sort('id')
)
df

id,fasta_id,dna_fasta_id,yield_binary,uniprot_id,taxon_id,gene_id,vector_seq,construct_id,expression_id,yield_cat,protein_seq,dna_seq
str,str,str,bool,str,i64,str,str,str,str,i64,str,str
"""SGC_S_000000""","""SGC_S_000000""","""SGC_S_000000""",true,"""P36575""",9606,"""ARR3""","""TAATACGACTCACTATAGGGGAATTGTGAG…","""ARR3A-c007""","""ARR3A-e015""",2,"""MHHHHHHSSGVDLGTENLYFQSMKVFKKTS…","""ATGCACCATCATCATCATCATTCTTCTGGT…"
"""SGC_S_000001""","""SGC_S_000001""","""SGC_S_000001""",true,"""P36575""",9606,"""ARR3""","""TAATACGACTCACTATAGGGGAATTGTGAG…","""ARR3A-c008""","""ARR3A-e016""",2,"""MHHHHHHSSGVDLGTENLYFQSMKVFKKTS…","""ATGCACCATCATCATCATCATTCTTCTGGT…"
"""SGC_S_000002""","""SGC_S_000002""","""SGC_S_000002""",true,"""Q96B67""",9606,"""ARRDC3""","""TAATACGACTCACTATAGGGGAATTGTGAG…","""ARRDC3A-c001""","""ARRDC3A-e006""",2,"""MHHHHHHSSGVDLGTENLYFQSMVLGKVKS…","""ATGCACCATCATCATCATCATTCTTCTGGT…"
"""SGC_S_000003""","""SGC_S_000003""","""SGC_S_000003""",true,"""Q96B67""",9606,"""ARRDC3""","""TAATACGACTCACTATAGGGGAATTGTGAG…","""ARRDC3A-c002""","""ARRDC3A-e007""",2,"""MHHHHHHSSGVDLGTENLYFQSMVLGKVKS…","""ATGCACCATCATCATCATCATTCTTCTGGT…"
"""SGC_S_000004""","""SGC_S_000004""","""SGC_S_000004""",false,"""Q96B67""",9606,"""ARRDC3""","""TAATACGACTCACTATAGGGGAATTGTGAG…","""ARRDC3A-c003""","""ARRDC3A-e008""",0,"""MHHHHHHSSGVDLGTENLYFQSMVLGKVKS…","""ATGCACCATCATCATCATCATTCTTCTGGT…"
…,…,…,…,…,…,…,…,…,…,…,…,…
"""SGC_S_014986""","""SGC_S_014986""","""SGC_S_014986""",false,"""O14512""",9606,"""SOCS7""","""TAATACGACTCACTATAGGGGAATTGTGAG…","""SOCS7A-c001""","""SOCS7A-e001""",0,"""MHHHHHHHHHHDLGTENLYFQSMPQHLQCP…","""ATGCACCATCATCATCATCATCACCATCAT…"
"""SGC_S_014987""","""SGC_S_014987""","""SGC_S_014987""",true,"""P12931""",9606,"""SRC""","""TAATACGACTCACTATAGGGGAATTGTGAG…","""SRCA-c001""","""SRCA-e001""",3,"""MHHHHHHHHHHDLGTENLYFQSMIQAEEWY…","""ATGCACCATCATCATCATCATCACCATCAT…"
"""SGC_S_014988""","""SGC_S_014988""","""SGC_S_014988""",true,"""Q9ULZ2""",9606,"""STAP1""","""TAATACGACTCACTATAGGGGAATTGTGAG…","""STAP1A-c001""","""STAP1A-e001""",2,"""MHHHHHHHHHHDLGTENLYFQSMDYVDVLN…","""ATGCACCATCATCATCATCATCACCATCAT…"


In [7]:

def get_taxon_name(taxon_id: int) -> str | None:
    url = f"https://rest.uniprot.org/taxonomy/{taxon_id}.json"
    headers = {'User-Agent': 'Python code generated by Claude'}
    try:
        resp = session.get(url, headers=headers, timeout=15)
        resp.raise_for_status()
        data = resp.json()
        return data.get('scientificName')
    except Exception as e:
        print(f"Error fetching taxon {taxon_id}: {e}")
        return None

taxon_ids = df.select(pl.col.taxon_id.unique())['taxon_id'].to_list()
taxon_id_to_name = {tid: get_taxon_name(tid) for tid in taxon_ids}
taxon_id_to_name


{5664: 'Leishmania major',
 5694: 'Trypanosoma equiperdum',
 5702: 'Trypanosoma brucei brucei',
 5821: 'Plasmodium berghei',
 5843: 'Plasmodium falciparum (isolate NF54)',
 8364: 'Xenopus tropicalis',
 9430: 'Desmodus rotundus',
 9483: 'Callithrix jacchus',
 9515: 'Sapajus apella',
 9541: 'Macaca fascicularis',
 9544: 'Macaca mulatta',
 9595: 'Gorilla gorilla gorilla',
 9598: 'Pan troglodytes',
 9601: 'Pongo abelii',
 9606: 'Homo sapiens',
 9669: 'Mustela putorius furo',
 9685: 'Felis catus',
 9713: 'Leptonychotes weddellii',
 9796: 'Equus caballus',
 9913: 'Bos taurus',
 9986: 'Oryctolagus cuniculus',
 10029: 'Cricetulus griseus',
 10090: 'Mus musculus',
 11103: 'Hepacivirus hominis',
 30522: 'Bos indicus x Bos taurus',
 37293: 'Aotus nancymaae',
 51337: 'Jaculus jaculus',
 57439: 'Upupa epops',
 61853: 'Nomascus leucogenys',
 69293: 'Gasterosteus aculeatus',
 75743: 'Scyliorhinus torazame',
 77932: 'Diceros bicornis minor',
 113115: 'Rhinopomastus cyanomelas',
 126794: 'Vaccinia viru

In [8]:
expert_metadata = pl.read_excel(
    '~/Documents/nicola_paper_2026/Supplementary Sheet S2_26052026.xlsx',
    sheet_name='E. coli', has_header=False, 
).transpose(column_names='column_1').select('ID', 'Description', 'Importance', 'What to capture')
expert_metadata

ID,Description,Importance,What to capture
str,str,str,str
"""T1""",""" Protein name (designed const…","""Critical""","""Protein name according to UniP…"
"""T2""","""Gene name (HGNC) ""","""Highly Enabling""","""Gene name according to HGNC na…"
"""T3""","""Gene species (NCBI)""","""Optional""","""Taxonomy according to NCBI (e.…"
"""T4""","""UniProt ID ""","""Optional/ highly enabling""","""UniProt identifier e.g. P00918"""
"""T5""","""Predicted protein localisation…","""Highly Enabling""","""Intracellular, membrane-bound,…"
…,…,…,…
"""Q2""","""Final batch - aggregation stat…","""Highly Enabling""","""Monodisperse, broad, multimeri…"
"""Q3""","""Final batch - stability / ther…","""Optional""","""Thermal unfolding as assessed …"
"""Q4""","""Final batch - identity and-or …","""Highly Enabling""","""Has size of protein been confi…"


In [9]:

def get_uniprot_data(uniprot_id: str) -> tuple[str, str | None]:
    """Fetch subcellular localization from UniProt REST API."""
    if not uniprot_id:
        return uniprot_id, None
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.json"
    headers = {'User-Agent': 'Python/generated by Claude'}
    try:
        resp = session.get(url, headers=headers, timeout=15)
        resp.raise_for_status()
        data = resp.json()
        uniprot_kb_id = data['uniProtkbId']
        locations = [
            subloc.get('location', {}).get('value')
            for comment in data.get('comments', [])
            if comment.get('commentType') == 'SUBCELLULAR LOCATION'
            for subloc in comment.get('subcellularLocations', [])
        ]
        locations = [l for l in locations if l]
        return uniprot_id, uniprot_kb_id, '; '.join(sorted(set(locations))) if locations else None
    except Exception as e:
        print(f"Error fetching {uniprot_id}: {e}")
        return uniprot_id, None

if uniprot_localisation_df_file.is_file():
    print(f"Reading uniprot df from {uniprot_localisation_df_file}")
    uniprot_df = pl.read_parquet(uniprot_localisation_df_file)
else:
    print(f"Loading uniprot df from the REST API")
    uniprot_ids = df.select(pl.col.uniprot_id.unique())['uniprot_id'].to_list()
    uniprot_data = []
    for uid in uniprot_ids:
        _, uniprot_kb_id, loc = get_uniprot_data(uid)
        uniprot_data.append((uid, uniprot_kb_id, loc))
        print(f"{uid}: {uniprot_kb_id}, {loc}")
    uniprot_df = pl.DataFrame(uniprot_data, schema={'uniprot_id': pl.String(), 'uniprot_name': pl.String(), 'localisation': pl.String()}, orient='row')
    uniprot_df.write_parquet(uniprot_localisation_df_file)


Loading uniprot df from the REST API
P41218: MNDA_HUMAN, Cytoplasm; Nucleus
Q9UBK8: MTRR_HUMAN, Cytoplasm
Q9H497: TOR3A_HUMAN, Cytoplasm; Endoplasmic reticulum lumen
Q00994: BEX3_HUMAN, Cytoplasm, cytosol; Nucleus
Q86YJ6: THNS2_HUMAN, Secreted
O14512: SOCS7_HUMAN, Cell membrane; Cytoplasm; Nucleus
Q8N1C8: Q8N1C8_HUMAN, None
Q9NWZ3: IRAK4_HUMAN, Cytoplasm
P09769: FGR_HUMAN, Cell membrane; Cell projection, ruffle membrane; Cytoplasm, cytoskeleton; Cytoplasm, cytosol; Mitochondrion inner membrane; Mitochondrion intermembrane space
O15523: DDX3Y_HUMAN, Cytoplasm; Nucleus
P27986: P85A_HUMAN, Cytoplasm
Q6PL18: ATAD2_HUMAN, Nucleus
Q92835: SHIP1_HUMAN, Cell membrane; Cytoplasm; Cytoplasm, cytoskeleton; Membrane; Membrane raft
Q9NRJ4: TULP4_HUMAN, Cytoplasm
O00154: BACH_HUMAN, Cytoplasm, cytosol; Mitochondrion
Q9BUB4: ADAT1_HUMAN, None
Q9UM47: NOTC3_HUMAN, Cell membrane; Nucleus
Q9NUL7: DDX28_HUMAN, Mitochondrion; Mitochondrion matrix; Mitochondrion matrix, mitochondrion nucleoid; Nucleus
B4DI

In [10]:
df_tags = (
    df.join(alignments_df, on='fasta_id', how='inner')
    .filter(~pl.col.subj_id.is_in(['<Start_M>', '<unaligned>', '<TEV>', '<SG_Link>', '<VDLG>', '<Thrombin>']))
    .group_by('id', 'subj_id')
    .agg(
        pl.col.uniprot_id.first(),
        pl.col.aln_subj_start.min(),
        pl.col.aln_subj_end.max(),
        pl.col.aln_start.min(),
        pl.col.ident_count.sum()
    )
    .sort('id', 'aln_start')
)
id_to_name = {}
name_components = []
current_id = ''
re_angles = re.compile('^<|>$')
for r in df_tags.rows(named=True):
    if r['id'] != current_id:
        if current_id != '':
            id_to_name[current_id] = '-'.join(name_components)
        name_components = []
        current_id = r['id']
    if r['subj_id'] == r['uniprot_id']:
        uniprot_name = uniprot_df.filter(uniprot_id=r['uniprot_id']).select('uniprot_name')[0,0]
        uniprot_name += f"[{r['aln_subj_start']+1}-{r['aln_subj_end']}]"
        name_components.append(uniprot_name)
    elif r['subj_id'] == '<His>':
        name_components.append(f"{r['ident_count']}xHis")
    elif r['subj_id'] == '<6His-link2>':
        name_components.append(f"6xHis")
    else:
        name_components.append(re_angles.sub('', r['subj_id']))

id_to_name[current_id] = '-'.join(name_components)
        


In [11]:
sgc_stockholm_expert_raw=pl.DataFrame({
    # 'id': df['id'],
    'T1': df['id'].replace_strict(id_to_name),
    'T2': df['gene_id'],
    'T3': df['taxon_id'].replace_strict(taxon_id_to_name),
    'T4': df['uniprot_id'],
    'C1': df['dna_seq'],
    'C2': df['protein_seq'],
    'C5': df['vector_seq'],
    'P7': df['yield_cat'].replace_strict({
        0: 'none',
        1: 'low',
        2: 'medium',
        3: 'high',
        4: 'very high'
    }),
    'P4': df['yield_binary'].cast(pl.String)
}).with_columns(
    C4=pl.lit('pNIC28-Bsa4'),
    E1=pl.lit('BL21-Gold(DE3) (Stratagene)'),
    E2=pl.lit('Terrific Broth (TB) (Formedium) + 50 μg/mL kanamycin + 34 μg/mL chloramphenicol'),
    E3=pl.lit('1 ml'),
    E4=pl.lit('18°C'),
    E5=pl.lit('pRARE2 plasmid from the Rosetta2 strain (Novagen)'),
    E6=pl.lit('700 rpm, 70 vertical pulses per minute in a small radius shaker (Glas-Col Vertiga)'),
    E7=pl.lit('96-well square deep-well plates (ABgene), well volume = 2.2 mL'),
    E8=pl.lit('overnight'),
    EB1=pl.lit('37°C'),
    EB2=pl.lit('2'),
    EB3=pl.lit('IPTG 0.5mM'),
    E10=pl.lit('detergents and freeze-thaw'),
    E11=pl.lit('pH 8.0; BUFF HEPES 100mM; '
               'SALT NaCl, 500mM; '
               'GLY 10%; '
               'RED TCEP, 0.5mM; '
               'OTHER imidazole, 10mM; '
               'OTHER lysozyme from chicken egg-white (Fluka), 1 mg/mL; '
               'OTHER Roche Complete EDTA-free protease inhibitor cocktail, 1 tablet/100 mL; '
               'OTHER n-dodecyl β-d-maltoside (DDM), 0.1%; '
               'OTHER MgSO4, 1mM; '
               'OTHER Merk benzonase, 125 U/mL'),
    P6=pl.lit('Manual estimation based on relative SDS-PAGE band thickness'),
    P1=pl.lit('Ni-NTA IMAC'),
    P2=pl.lit('pH 7.5; BUFF HEPES 20mM; '
              'SALT NaCl, 500mM; '
              'GLY 10%; '
              'OTHER imidazole, 500mM; '
              'RED TCEP, 0.5mM; '),
    P3=pl.lit('Manual estimation based on relative SDS-PAGE band thickness'),
    O1=pl.lit("""
{
    'cell_line_link_addgene':'https://www.addgene.org/26242/', 
    'vector_link_genebank': 'https://www.ncbi.nlm.nih.gov/nuccore/EF198106.1',
    'vector_link_addgene': 'https://www.addgene.org/26103/',
    'publication_doi' : '10.1016/j.pep.2007.11.008'
}
"""),
    O2=pl.lit('7')
).join(
    uniprot_df, left_on='T4', right_on='uniprot_id', how='inner', coalesce=True
).rename({'localisation': 'T5'})

In [12]:
sgc_stockholm_expert = pl.concat([
    expert_metadata.transpose(column_names='ID'), 
    sgc_stockholm_expert_raw
], how='diagonal').select(expert_metadata['ID'].to_list())
sgc_stockholm_expert


T1,T2,T3,T4,T5,T6,T7,CX1,CX2,CX3,C1,C2,C3,C4,C5,C6,E1,E2,E3,E4,E5,E6,E7,E8,E9,E10,E11,E12,E13,EB1,EB2,EB3,P1,P2,P3,P4,P5,P6,P7,P8,P9,P10,P11,P12,P13,P14,P15,Q1,Q2,Q3,Q4,O1,O2
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
""" Protein name (designed const…","""Gene name (HGNC) ""","""Gene species (NCBI)""","""UniProt ID ""","""Predicted protein localisation…","""PDB id""","""PTMs""","""Complex ID ""","""Complex name ""","""Complex formation method ""","""DNA sequence of coding region …","""Amino acid sequence including …","""Nucleotide sequence transcript""","""Vector name ""","""Vector sequence ""","""Annotated sequence""","""Host strain or cell line""","""Culture medium name or source""","""Culture volume ""","""Culture conditions - temperatu…","""Culture conditions-additives""","""Culture conditions - rpm""","""Culture conditions - type of g…","""Growth time ""","""Pellet weight ""","""Lysis method ""","""Lysis buffer""","""Total expression determination…","""Total expression - is the prot…","""Culture conditions - temperatu…","""OD600 at induction ""","""Culture conditions - inductio…","""First-step purification method""","""First-step purification buffer…","""First-step purification binary…","""First-step binary purification…","""First-step purification protei…","""First-step purification yield …","""First-step purification yield ""","""Second-step purification metho…","""Second-step purification yield…","""Second-step purification yield…","""Third-step purification method…","""Third-step purification yield …","""Third-step purification yield ""","""Tag removal step""","""Final protein buffer""","""Final batch - purity ""","""Final batch - aggregation stat…","""Final batch - stability / ther…","""Final batch - identity and-or …","""Other comments""","""Data record completeness score"""
"""Critical""","""Highly Enabling""","""Optional""","""Optional/ highly enabling""","""Highly Enabling""","""Highly Enabling""","""Highly Enabling""","""Mandatory for complexes""","""Mandatory for complexes""","""Highly Enabling for Complexes""","""Critical""","""Critical""","""Highly Enabling""","""Highly Enabling""","""Critical""","""Optional""","""Critical""","""Critical""","""Critical""","""Critical""","""Optional""","""Critical""","""Highly Enabling""","""Highly Enabling""","""Highly Enabling""","""Highly Enabling""","""Highly Enabling""","""Highly Enabling""","""Highly Enabling""","""Critical""","""Highly Enabling""","""Highly Enabling""","""Critical""","""Highly Enabling""","""Highly Enabling""","""Critical""","""Highly Enabling""","""Highly Enabling""","""Critical""","""Highly Enabling""","""Highly Enabling""","""Highly Enabling""","""Optional""","""Highly Enabling""","""Optional""","""Optional""","""Highly Enabling""","""Highly Enabling""","""Highly Enabling""","""Optional""","""Highly Enabling""","""Optional""","""Highly Enabling"""
"""Protein name according to UniP…","""Gene name according to HGNC na…","""Taxonomy according to NCBI (e.…","""UniProt identifier e.g. P00918""","""Intracellular, membrane-bound,…","""PDB identifier""","""Mapped post-translational modi…","""An id to link different record…","""Descriptive name of the comple…","""Polycistronic construct, or mu…","""Nucleotide sequence from start…","""Amino acid sequence from start…","""Nucleotide sequence from trans…","""Vector name excluding insert (…","""Nucleotide sequence of entire …","""Vector or insert sequence with…","""Name of E. coli strain used (e…","""Name and composition of medium…","""ml to L""","""Temperature used post-inductio…","""Any additional co-factors or c…","""Shaker frequency used during g…","""Type and volume of growth cont…","""Hours to days""","""Wet weight of pellet in g""","""How cells were lysed for examp…","""Basic components in buffer sys…","""Experimental procedure used to…","""

In [13]:

def resolve(filename) -> pathlib.Path:
    if type(filename) == str:
        filename = pathlib.Path(filename)
    if str(filename).startswith('~'):
        filename = filename.expanduser()
    return filename.resolve()

def write_fasta(sequences:typing.Iterable[typing.Tuple[str, str]]|typing.Mapping[str, str], filename:str|os.PathLike, seq_chunk_size:int=80):
    """
    Write a sequence of (id, sequence) tuples to a FASTA file.
    :param sequences: A sequence of (id, sequence) tuples.
    :param filename: The name of the file to write to.
    :param seq_chunk_size: The size of each sequence chunk in the output file.
    """
    if isinstance(sequences, typing.Mapping):
        sequences = sequences.items()
    filename = resolve(filename)
    is_gzip = filename.suffix == '.gz'
    with gzip.open(filename, 'wt') if is_gzip else open(filename, 'w') as f:
        for seq_id, sequence in sequences:
            f.write(f">{seq_id}\n")
            for i in range(0, len(sequence), seq_chunk_size):
                f.write(sequence[i:i + seq_chunk_size] + '\n')

In [14]:
# sgc_stockholm_expert.write_excel('~/Documents/nicola_paper_2026/sgc_stockholm_expert.xlsx')

In [15]:
sgc_stockholm_expert_raw.select(pl.col.C1.n_unique())

C1
u32
11023


In [16]:
sgc_stockholm_expert_raw.shape

(14676, 31)

In [17]:
assert sgc_stockholm_expert_raw.group_by('C2').agg(pl.col.T1.n_unique()).max().select('T1').item() == 1
fasta_df = (
    sgc_stockholm_expert_raw.with_row_index("row_idx")
    .group_by('C2')
    .agg(fasta_id=pl.col.T1.first() + pl.lit('-') + (pl.col.row_idx.first() + 1).cast(pl.String))
)
write_fasta(zip(fasta_df['fasta_id'], fasta_df['C2']), 
            '/Users/evgeny/Documents/nicola_paper_2026/biostudies/sgc_stockholm_expert.fasta.gz')
